# 00 · Antes de empezar: comprobar el terreno

**Objetivo:** salir de esta libreta sabiendo *qué modelo vas a usar y por qué*, y con la
certeza de que tu entorno soporta **tool calling**, que es la pieza sin la cual nada de
lo que viene después funciona.

## Por qué dedicamos diez minutos a esto

Un modelo puede aparecer en `ollama list` y estar perfectamente inservible. No es
un caso teórico: al preparar este seminario, en esta misma máquina, nos encontramos con

| modelo listado | lo que pasa de verdad |
|---|---|
| `qwen3.5:cloud` | `was retired at 2026-09-25` — el proveedor lo retiró |
| `glm-5.2:cloud` | `not included in your free usage` — fuera del plan gratuito |
| `gemma4:31b-cloud` | funciona, y además hace tool calling |

Los catálogos de modelos **caducan**. Por eso en este seminario no escribimos un nombre
de modelo a mano en cuarenta celdas: lo elegimos **una vez**, comprobando que responde,
y todo lo demás lo hereda.

## 0.1 ¿Qué tengo instalado?

Lo primero es saber sobre qué estamos construyendo. Estas son las versiones que
usa el seminario; si las tuyas son muy distintas, algunas rutas de import habrán cambiado.

In [1]:
import platform
import sys
import importlib.metadata as md

print("Python:", sys.version.split()[0], "|", platform.system(), platform.machine())
print()
PAQUETES = [
    "langgraph", "langgraph-prebuilt", "langgraph-checkpoint",
    "langchain", "langchain-core", "langchain-openai",
    "openai", "pydantic",
]
for paquete in PAQUETES:
    try:
        print(f"  {paquete:22} {md.version(paquete)}")
    except md.PackageNotFoundError:
        print(f"  {paquete:22} (NO instalado)")

import comun
print()
print("Daemon de Ollama :", comun.OLLAMA)
print("Endpoint OpenAI  :", comun.BASE_URL_OPENAI)


Python: 3.12.13 | Linux x86_64

  langgraph              1.2.12
  langgraph-prebuilt     1.1.0
  langgraph-checkpoint   4.2.0
  langchain              1.4.3
  langchain-core         1.6.6
  langchain-openai       1.6.7
  openai                 3.24.0
  pydantic               2.13.5

Daemon de Ollama : http://localhost:11434
Endpoint OpenAI  : http://localhost:11434/v1


## 0.2 ¿Qué modelos tengo?

Esta lista es lo que Ollama **dice** que tienes. Tómalo como una pista, no como una
garantía: en el paso siguiente comprobamos cuáles responden de verdad.

In [2]:
from comun import modelos_listados

listados = modelos_listados()
print(f"Tienes {len(listados)} modelos:\n")
for nombre in listados:
    etiqueta = "nube" if "cloud" in nombre else "local"
    print(f"  [{etiqueta:5}] {nombre}")


Tienes 24 modelos:

  [nube ] gemma4:cloud
  [nube ] minimax-m2.7:cloud
  [nube ] nemotron-3-super:cloud
  [nube ] minimax-m3:cloud
  [nube ] kimi-k3:cloud
  [nube ] gpt-oss:20b-cloud
  [nube ] kimi-k2.6:cloud
  [nube ] kimi-k2.7-code:cloud
  [nube ] gpt-oss:120b-cloud
  [nube ] glm-5.2:cloud
  [nube ] deepseek-v4-pro:0813-cloud
  [nube ] glm-5.3:cloud
  [nube ] nemotron-3-nano:30b-cloud
  [nube ] gemma4:31b-cloud
  [nube ] glm-5.3-flash:cloud
  [nube ] mistral-large-3:675b-cloud
  [nube ] deepseek-v4.1-flash:cloud
  [nube ] nemotron-3-ultra:cloud
  [nube ] qwen3.5:cloud
  [nube ] qwen3.5:397b-cloud
  [local] qwen2.5vl:3b
  [local] minicpm-v:latest
  [local] gemma4:e4b-32k
  [local] gemma4:e4b


## 0.3 La prueba de fuego: ¿cuál responde?

Aquí está la lección más importante de la libreta. Vamos a **enviarle un mensaje a cada
modelo** y a quedarnos con el motivo exacto del fallo cuando lo haya. Son los errores
reales de esta máquina, con su texto literal.

Sondear un modelo local la primera vez tarda: hay que cargarlo en memoria. Es normal.

In [3]:
from comun import probar_modelo

# Sondeamos los de nube (fallan rápido) y el candidato local.
a_probar = list(dict.fromkeys([m for m in listados if "cloud" in m] + ["gemma4:e4b"]))

print(f"{'modelo':26} {'¿responde?':11} detalle")
print("-" * 104)
sondeos = {}
for nombre in a_probar:
    ok, detalle = probar_modelo(nombre)
    sondeos[nombre] = (ok, detalle)
    print(f"{nombre:26} {('SÍ' if ok else 'NO'):11} {detalle[:64]}")


modelo                     ¿responde?  detalle
--------------------------------------------------------------------------------------------------------


gemma4:cloud               SÍ          responde correctamente
minimax-m2.7:cloud         NO          HTTP 402: {"error":"This model is not in the Free plan. Pro is $


nemotron-3-super:cloud     SÍ          responde correctamente


minimax-m3:cloud           NO          HTTP 402: {"error":"This model is not in the Free plan. Pro is $
kimi-k3:cloud              NO          HTTP 402: {"error":"This model is not in the Free plan. Pro is $


gpt-oss:20b-cloud          SÍ          responde correctamente


...

gemma4:31b-cloud           SÍ          responde correctamente
glm-5.3-flash:cloud        NO          HTTP 402: {"error":"This model is not in the Free plan. Pro is $


mistral-large-3:675b-cloud NO          HTTP 402: {"error":"This model is not in the Free plan. Pro is $
deepseek-v4.1-flash:cloud  NO          HTTP 402: {"error":"This model is not in the Free plan. Pro is $


nemotron-3-ultra:cloud     SÍ          responde correctamente
qwen3.5:cloud              NO          HTTP 410: {"error":"qwen3.5:397b was retired at 2026-09-25 00:00


qwen3.5:397b-cloud         NO          HTTP 410: {"error":"qwen3.5:397b was retired at 2026-09-25 00:00


gemma4:e4b                 SÍ          responde correctamente


### No todos los fallos son iguales

El mensaje de error es la información más valiosa de todo el preflight, porque **cada
causa exige una decisión distinta**.

In [4]:
retirados = [m for m, (ok, detalle) in sondeos.items() if "retired" in detalle]
sin_cuota = [m for m, (ok, detalle) in sondeos.items() if "free usage" in detalle]
sanos = [m for m, (ok, _) in sondeos.items() if ok]

print(f"Modelos que RESPONDEN              : {len(sanos)}")
print(f"Modelos RETIRADOS por el proveedor : {len(retirados)}   {retirados}")
print(f"Modelos fuera del plan gratuito    : {len(sin_cuota)}")
print()
print("Tres finales distintos, tres causas distintas:")
print("  retirado  -> el proveedor lo quitó: hay que cambiar de modelo, no pagar")
print("  sin cuota -> existe y funciona, pero no entra en tu plan: es dinero")
print("  sano      -> adelante")


Modelos que RESPONDEN              : 7
Modelos RETIRADOS por el proveedor : 2   ['qwen3.5:cloud', 'qwen3.5:397b-cloud']
Modelos fuera del plan gratuito    : 0

Tres finales distintos, tres causas distintas:
  retirado  -> el proveedor lo quitó: hay que cambiar de modelo, no pagar
  sin cuota -> existe y funciona, pero no entra en tu plan: es dinero
  sano      -> adelante


## 0.4 Elegir el modelo del seminario

`comun.elegir_modelo()` recorre los candidatos en orden de preferencia y devuelve el
primero que **contesta de verdad**:

1. lo que digas en la variable de entorno `SEMINARIO_MODELO` (tu elección manda),
2. `gemma4:31b-cloud` — nube gratuita,
3. `gemma4:e4b` — local, sin necesidad de internet.

Fíjate en el detalle: **no basta con estar en la lista**. Se comprueba respondiendo.

In [5]:
from comun import elegir_modelo, es_nube

MODELO = elegir_modelo()
print()
print("Modelo del seminario:", MODELO)
print("¿Vive en la nube?    ", es_nube(MODELO),
      "-> si es False, todo funciona sin conexión")


[preflight] OK   gemma4:31b-cloud         responde correctamente
[preflight] Modelo activo: gemma4:31b-cloud

Modelo del seminario: gemma4:31b-cloud
¿Vive en la nube?     True -> si es False, todo funciona sin conexión


## 0.5 ¿Responde bien a una petición normal?

Comprobación trivial de que la tubería completa funciona: LangChain → endpoint
compatible con OpenAI → daemon de Ollama → modelo (en la nube o en disco).

In [6]:
import comun

modelo = comun.llm()
respuesta = modelo.invoke("Responde únicamente con la palabra: listo")
print("respuesta del modelo:", respuesta.content.strip()[:120])


respuesta del modelo: listo


## 0.6 La prueba crítica: **¿sabe usar herramientas?**

Todo el seminario se apoya en esto. Un agente es, en el fondo, un modelo que puede
pedir que ejecutemos funciones. Si el modelo no es capaz de emitir `tool_calls`,
ningún grafo que escribamos servirá de nada.

Le damos una herramienta y le pedimos explícitamente que la use. Si no la pide, paramos.
Esta es la diferencia entre enterarse en el minuto 5 o descubrirlo en el minuto 150.

In [7]:
from langchain.tools import tool
from comun import seccion

@tool
def buscar_precio(producto: str) -> float:
    """Devuelve el precio estimado de un producto en EUR."""
    return 5.0

seccion("PRUEBA CRÍTICA: ¿sabe este modelo usar herramientas?")

respuesta = comun.llm().bind_tools([buscar_precio]).invoke(
    "¿Cuánto cuesta el pollo? Usa la herramienta buscar_precio."
)

print("tool_calls solicitados:", respuesta.tool_calls or "(ninguno)")
print()

assert respuesta.tool_calls, (
    "Este modelo NO ha pedido ninguna herramienta.\n"
    "Sin tool calling no hay agentes: cambia de modelo antes de seguir.\n"
    "Prueba con:  ollama pull gemma4:e4b\n"
    "y luego:     SEMINARIO_MODELO=gemma4:e4b  (o edita comun.CANDIDATOS)"
)
print("Tool calling confirmado. Todo listo para empezar.")



PRUEBA CRÍTICA: ¿sabe este modelo usar herramientas?


tool_calls solicitados: [{'name': 'buscar_precio', 'args': {'producto': 'pollo'}, 'id': 'call_2xxecko7', 'type': 'tool_call'}]

Tool calling confirmado. Todo listo para empezar.


---

## Si algo ha fallado

**El daemon no responde** (`No puedo hablar con Ollama`). Arranca el servicio:

```bash
ollama serve          # en otra terminal
```

**Ningún modelo contesta.** Descarga uno local, que además es gratis y sin cuota:

```bash
ollama pull gemma4:e4b
```

**Los modelos de nube dan error de cuota o de modelo retirado.** Es lo normal y no es
culpa tuya: el plan gratuito cambia. Dos salidas:

* usar el modelo local `gemma4:e4b` (hace tool calling, verificado),
* o crear una API key en <https://ollama.com/settings/keys> y exportarla:

```bash
export OLLAMA_API_KEY="tu_clave"
```

**Quiero forzar un modelo concreto** sin tocar código:

```bash
SEMINARIO_MODELO=gemma4:e4b jupyter lab
```

**Las peticiones se quedan esperando o dan `429`.** El plan gratuito permite **una sola
petición concurrente** (y una cuota mensual pequeña). Dos consecuencias para el taller:

* lanza los ejemplos **en secuencia**, no en paralelo: nada de *fan-out* entre varios
  agentes a la vez;
* si se agota la cuota, el retroceso de verdad es **local** (`gemma4:e4b`), no otro
  modelo de nube: la cuota es de la cuenta, no del modelo.

Recuerda también que los términos de Ollama piden ser mayor de edad y **una cuenta por
persona**: cada alumno usa su propio *signin*, no una clave compartida de clase.

### Lo que hemos aprendido y seguiremos usando

| Idea | Por qué importa |
|---|---|
| Un modelo listado no es un modelo que funciona | Los catálogos caducan y las cuotas cambian |
| Hay que comprobar **respondiendo**, no leyendo una lista | Es la única prueba fiable |
| El tool calling es el requisito duro | Sin él no hay agentes, solo chat |
| El modelo se elige en un sitio | Cambiar de modelo no debe tocar 40 celdas |